In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
import time
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from langchain_typesafe import TypeSafeClassifier, Choice
import pandas as pd

In [2]:
load_dotenv()

True

In [3]:
dataset = pd.read_csv(r"data/golden-dataset.csv")
output_file = "classification_results.json"
dataset.columns

Index(['id', 'query', 'level', 'domain', 'intent_type', 'intent', 'intents'], dtype='str')

In [4]:
dataset['intent'].value_counts()

intent
multiple                30
out_of_scope            15
employee_information    11
timesheet               11
leaves                  11
benefits                11
payroll                 11
Name: count, dtype: int64

In [5]:
# Initialize classifier
classifier = TypeSafeClassifier()

# Define bounded questions for System One evaluation
questions = {
    "intent_type": Choice(
        instructions=(
            "Classify whether this inquiry requires handling within a single HR domain, "
            "requires coordinating across two or more distinct HR domains, or falls "
            "entirely outside human resources operations."
        ),
        criteria={
            "single": (
                "The core request can be fully resolved within ONE functional domain "
                "(Employee Information, Leaves, Timesheets, Benefits, or Payroll). "
                "Conversational context, background history, or mentioning multiple facts "
                "within the same subject still counts as single intent."
            ),
            "multiple": (
                "The query explicitly demands status checks, calculations, or investigations "
                "across TWO OR MORE distinct HR domains (e.g., verifying a timesheet AND its "
                "impact on a paycheck, or checking leave quotas AND health benefit coverage)."
            ),
            "out_of_scope": (
                "The request does not belong to HR, time, leave, benefits, or payroll. "
                "Includes IT hardware or accounts (laptops, VPN, SSO, password resets, software licenses), "
                "physical facilities (office badges, HVAC, parking), corporate travel and expense per diems, "
                "legal templates or whistleblowing, sales assets, and general programming."
            ),
        },
    ),
    "intent": Choice(
        instructions=(
            "Identify the specific operational HR domain responsible for resolving this query. "
            "If the request requires multiple domains or is outside HR, classify it as 'multiple' "
            "or 'out_of_scope' respectively."
        ),
        criteria={
            "employee_information": (
                "Demographic records, worker profiles, job titles, department assignments, "
                "manager/reporting structure, start dates, and active or inactive employment standing."
            ),
            "leaves": (
                "Paid time off (PTO), vacation days, sick time, parental/bereavement leave, "
                "absence approval status, accrual bank balances, and company leave rollover policies."
            ),
            "timesheet": (
                "Shift hours worked, timecard submissions, manager punch approvals, missing timecards, "
                "clock-in/out corrections, and timecard exception flags."
            ),
            "benefits": (
                "Health, dental, and vision insurance plans, coverage tiers, deductibles, qualifying "
                "life events (marriage, birth, dependents), open enrollment, and 401(k) plan matches."
            ),
            "payroll": (
                "Paychecks, salary direct deposit settlement, gross/net pay variances, pay run cycles, "
                "retroactive adjustments, tax withholdings, and wage garnishments."
            ),
            "multiple": (
                "Cross-domain requests that span two or more of the above domains simultaneously "
                "and cannot be isolated to a single department worker."
            ),
            "out_of_scope": (
                "Non-HR corporate operations including IT service desk, computer hardware, network/VPN, "
                "building facilities, corporate travel booking, sales operations, or software development."
            ),
        },
    ),
}

C:\Users\agadi\AppData\Local\Temp\ipykernel_19996\3780960238.py:2: LangChainBetaWarning: The class `TypeSafeClassifier` is in beta. It is actively being worked on, so the API may change.
  classifier = TypeSafeClassifier()


In [6]:
results = []

# Run classification loop with progress tracking
pbar = tqdm(
    dataset.iterrows(),
    total=len(dataset),
    desc="Classifying queries",
    unit="query",
    dynamic_ncols=True,
)

for idx, row in pbar:
    query_text = str(row["query"]).strip()
    gold_intent_type = str(row.get("intent_type", "")).strip()
    gold_intent = str(row.get("intent", "")).strip()
    raw_intents = row.get("intents")
    gold_intents = (
        str(raw_intents).split("|")
        if pd.notna(raw_intents) and str(raw_intents).lower() != "none"
        else None
    )

    start_time = time.perf_counter()
    try:
        response = classifier.invoke(
            {
                "state": query_text,
                "questions": questions,
            }
        )
        latency_ms = (time.perf_counter() - start_time) * 1000

        pred_intent_type = response.choices["intent_type"].choice
        pred_intent_type_conf = response.choices["intent_type"].confidence
        pred_intent = response.choices["intent"].choice
        pred_intent_conf = response.choices["intent"].confidence

        pbar.set_postfix({"Latency": f"{latency_ms:.0f}ms"})

        results.append(
            {
                "id": row.get("id"),
                "query": query_text,
                "level": row.get("level"),
                "domain": row.get("domain"),
                "latency_ms": round(latency_ms, 2),
                "ground_truth": {
                    "intent_type": gold_intent_type,
                    "intent": gold_intent,
                    "intents": gold_intents,
                },
                "prediction": {
                    "intent_type": pred_intent_type,
                    "intent_type_confidence": round(pred_intent_type_conf, 4),
                    "intent": pred_intent,
                    "intent_confidence": round(pred_intent_conf, 4),
                },
                "details": response.model_dump(mode="json"),
            }
        )

    except Exception as e:
        tqdm.write(f"\n[ERROR] Row {idx} ({row.get('id', 'N/A')}): {e}")
        results.append(
            {
                "id": row.get("id"),
                "query": query_text,
                "level": row.get("level"),
                "domain": row.get("domain"),
                "error": str(e),
            }
        )

Classifying queries:   0%|          | 0/100 [00:00<?, ?query/s]

In [7]:
# Persist raw run output
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

# Execution telemetry summary
latencies = [r["latency_ms"] for r in results if "latency_ms" in r]
latencies.sort()
p50 = latencies[len(latencies) // 2] if latencies else 0
p95 = latencies[int(len(latencies) * 0.95)] if latencies else 0

print("\n" + "=" * 50)
print("BENCHMARK EXECUTION COMPLETE")
print("=" * 50)
print(f"Total Queries Processed : {len(results)}/{len(dataset)}")
print(f"Successful Invocations  : {len(latencies)}")
print(f"Failed Invocations      : {len(results) - len(latencies)}")


BENCHMARK EXECUTION COMPLETE
Total Queries Processed : 100/100
Successful Invocations  : 100
Failed Invocations      : 0


In [8]:
# =============================================================================
# Configuration
# =============================================================================

CM_PLOT_FILE = "benchmark_confusion_matrices.png"
ERRORS_FILE = "misclassified_queries.csv"
SUMMARY_FILE = "benchmark_summary.json"

TYPE_LABELS = [
    "single",
    "multiple",
    "out_of_scope",
]

INTENT_LABELS = [
    "employee_information",
    "leaves",
    "timesheet",
    "benefits",
    "payroll",
    "multiple",
    "out_of_scope",
]

SHORT_INTENT_LABELS = [
    "emp_info",
    "leaves",
    "timesheet",
    "benefits",
    "payroll",
    "multiple",
    "out_of_scope",
]

DIFFICULTY_ORDER = [
    "easy",
    "medium",
    "hard",
]


# =============================================================================
# 1. Load Benchmark Results
# =============================================================================

with open(output_file, "r", encoding="utf-8") as f:
     classification_results_json = f.read()

results = json.loads(classification_results_json)

if not isinstance(results, list):
    raise ValueError("Expected benchmark results to be a JSON list.")

valid_records = [
    r
    for r in results
    if isinstance(r, dict)
    and "error" not in r
    and "prediction" in r
    and "ground_truth" in r
]

print(
    f"Loaded {len(valid_records)}/{len(results)} "
    "valid inference records."
)

if not valid_records:
    raise ValueError("No valid benchmark records found.")

N = len(valid_records)


# =============================================================================
# 2. Intent Type Evaluation — Complexity Gate
# =============================================================================

y_true_type = [
    r["ground_truth"].get("intent_type")
    for r in valid_records
]

y_pred_type = [
    r["prediction"].get("intent_type")
    for r in valid_records
]

type_accuracy = accuracy_score(
    y_true_type,
    y_pred_type,
)

cm_type = confusion_matrix(
    y_true_type,
    y_pred_type,
    labels=TYPE_LABELS,
)

type_report = classification_report(
    y_true_type,
    y_pred_type,
    labels=TYPE_LABELS,
    digits=3,
    zero_division=0,
    output_dict=True,
)

print("\n" + "=" * 70)
print("1. INTENT TYPE — COMPLEXITY GATE")
print("=" * 70)

print(
    classification_report(
        y_true_type,
        y_pred_type,
        labels=TYPE_LABELS,
        digits=3,
        zero_division=0,
    )
)


# =============================================================================
# 3. Primary Intent Evaluation — Domain Routing
# =============================================================================

y_true_intent = [
    r["ground_truth"].get("intent")
    for r in valid_records
]

y_pred_intent = [
    r["prediction"].get("intent")
    for r in valid_records
]

intent_accuracy = accuracy_score(
    y_true_intent,
    y_pred_intent,
)

cm_intent = confusion_matrix(
    y_true_intent,
    y_pred_intent,
    labels=INTENT_LABELS,
)

intent_report = classification_report(
    y_true_intent,
    y_pred_intent,
    labels=INTENT_LABELS,
    digits=3,
    zero_division=0,
    output_dict=True,
)

print("\n" + "=" * 70)
print("2. PRIMARY INTENT — DOMAIN ROUTING")
print("=" * 70)

print(
    classification_report(
        y_true_intent,
        y_pred_intent,
        labels=INTENT_LABELS,
        digits=3,
        zero_division=0,
    )
)


# =============================================================================
# 4. Strict Exact Evaluation
# =============================================================================

type_matches = [
    true == pred
    for true, pred in zip(
        y_true_type,
        y_pred_type,
    )
]

intent_matches = [
    true == pred
    for true, pred in zip(
        y_true_intent,
        y_pred_intent,
    )
]

strict_matches = [
    type_match and intent_match
    for type_match, intent_match in zip(
        type_matches,
        intent_matches,
    )
]

strict_accuracy = np.mean(strict_matches)


print("\n" + "=" * 70)
print("3. OVERALL BENCHMARK ACCURACY")
print("=" * 70)

print(f"Total Queries              : {N}")
print(f"Intent Type Accuracy       : {type_accuracy:.1%}")
print(f"Primary Intent Accuracy    : {intent_accuracy:.1%}")
print(f"Strict Exact Accuracy      : {strict_accuracy:.1%}")


# =============================================================================
# 5. Accuracy Breakdown by Difficulty
# =============================================================================

df_records = pd.DataFrame(
    [
        {
            "id": r.get("id"),
            "level": r.get("level", "unknown"),
            "query": r.get("query", ""),
            "type_match": type_match,
            "intent_match": intent_match,
            "exact_match": exact_match,
        }
        for r, type_match, intent_match, exact_match in zip(
            valid_records,
            type_matches,
            intent_matches,
            strict_matches,
        )
    ]
)


def summarize_level(group):
    return pd.Series(
        {
            "total_queries": len(group),
            "intent_type_acc": group["type_match"].mean(),
            "domain_intent_acc": group["intent_match"].mean(),
            "strict_both_acc": group["exact_match"].mean(),
        }
    )


level_summary = (
    df_records
    .groupby("level", sort=False)
    .apply(summarize_level)
    .reset_index()
)

# Ensure easy → medium → hard ordering.
level_summary["level"] = pd.Categorical(
    level_summary["level"],
    categories=DIFFICULTY_ORDER,
    ordered=True,
)

level_summary = (
    level_summary
    .sort_values("level")
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("4. ACCURACY BY QUERY COMPLEXITY")
print("=" * 70)

display_summary = level_summary.copy()

display_summary["intent_type_acc"] = (
    display_summary["intent_type_acc"]
    .map(lambda x: f"{x:.1%}")
)

display_summary["domain_intent_acc"] = (
    display_summary["domain_intent_acc"]
    .map(lambda x: f"{x:.1%}")
)

display_summary["strict_both_acc"] = (
    display_summary["strict_both_acc"]
    .map(lambda x: f"{x:.1%}")
)

print(
    display_summary.to_string(index=False)
)


# =============================================================================
# 6. Latency Telemetry
# =============================================================================

latencies = [
    r.get("latency_ms")
    for r in valid_records
    if isinstance(r.get("latency_ms"), (int, float))
    and r.get("latency_ms") >= 0
]

latency_stats = {}

if latencies:

    latency_array = np.asarray(
        latencies,
        dtype=float,
    )

    latency_stats = {
        "count": len(latency_array),
        "p50_ms": float(np.percentile(latency_array, 50)),
        "p90_ms": float(np.percentile(latency_array, 90)),
        "p95_ms": float(np.percentile(latency_array, 95)),
        "p99_ms": float(np.percentile(latency_array, 99)),
        "mean_ms": float(np.mean(latency_array)),
        "std_ms": float(np.std(latency_array)),
        "min_ms": float(np.min(latency_array)),
        "max_ms": float(np.max(latency_array)),
    }

    print("\n" + "=" * 70)
    print("5. LATENCY PERFORMANCE PROFILE — WALL CLOCK")
    print("=" * 70)

    print(f"Samples      : {latency_stats['count']}")
    print(f"P50 Latency  : {latency_stats['p50_ms']:.1f} ms")
    print(f"P90 Latency  : {latency_stats['p90_ms']:.1f} ms")
    print(f"P95 Latency  : {latency_stats['p95_ms']:.1f} ms")
    print(f"P99 Latency  : {latency_stats['p99_ms']:.1f} ms")
    print(
        f"Mean Latency : "
        f"{latency_stats['mean_ms']:.1f} ms "
        f"± {latency_stats['std_ms']:.1f} ms"
    )
    print(f"Min Latency  : {latency_stats['min_ms']:.1f} ms")
    print(f"Max Latency  : {latency_stats['max_ms']:.1f} ms")


# =============================================================================
# 7. Export Misclassified Queries
# =============================================================================

errors = []

for r in valid_records:

    gold_type = r["ground_truth"].get("intent_type")
    pred_type = r["prediction"].get("intent_type")

    gold_intent = r["ground_truth"].get("intent")
    pred_intent = r["prediction"].get("intent")

    type_match = gold_type == pred_type
    intent_match = gold_intent == pred_intent

    if type_match and intent_match:
        continue

    errors.append(
        {
            "id": r.get("id"),
            "level": r.get("level"),
            "query": r.get("query"),

            "gold_type": gold_type,
            "pred_type": pred_type,
            "type_correct": type_match,
            "type_confidence": r["prediction"].get(
                "intent_type_confidence"
            ),

            "gold_intent": gold_intent,
            "pred_intent": pred_intent,
            "intent_correct": intent_match,
            "intent_confidence": r["prediction"].get(
                "intent_confidence"
            ),

            "type_error": (
                f"{gold_type} -> {pred_type}"
                if not type_match
                else None
            ),

            "intent_error": (
                f"{gold_intent} -> {pred_intent}"
                if not intent_match
                else None
            ),
        }
    )

errors_df = pd.DataFrame(errors)

errors_df.to_csv(
    ERRORS_FILE,
    index=False,
    encoding="utf-8",
)

print(
    f"\nExported {len(errors_df)} misclassified queries "
    f"to '{ERRORS_FILE}'."
)


# =============================================================================
# 8. Error Distribution
# =============================================================================

if not errors_df.empty:

    print("\n" + "=" * 70)
    print("6. ERROR DISTRIBUTION")
    print("=" * 70)

    type_errors = (
        errors_df.loc[
            ~errors_df["type_correct"],
            ["gold_type", "pred_type"],
        ]
        .value_counts()
        .rename("count")
    )

    if not type_errors.empty:
        print("\nIntent Type Errors:")
        print(type_errors.to_string())

    intent_errors = (
        errors_df.loc[
            ~errors_df["intent_correct"],
            ["gold_intent", "pred_intent"],
        ]
        .value_counts()
        .rename("count")
    )

    if not intent_errors.empty:
        print("\nPrimary Intent Errors:")
        print(intent_errors.to_string())


# =============================================================================
# 9. Save Machine-Readable Summary
# =============================================================================

summary = {
    "benchmark": {
        "total_records": len(results),
        "valid_records": N,
        "invalid_records": len(results) - N,
    },

    "accuracy": {
        "intent_type": float(type_accuracy),
        "primary_intent": float(intent_accuracy),
        "strict_exact": float(strict_accuracy),
    },

    "classification_reports": {
        "intent_type": type_report,
        "primary_intent": intent_report,
    },

    "difficulty": (
        level_summary
        .assign(
            level=lambda df: df["level"].astype(str)
        )
        .to_dict(orient="records")
    ),

    "latency": latency_stats,

    "errors": {
        "count": len(errors),
        "error_rate": (
            len(errors) / N
            if N
            else 0.0
        ),
    },
}

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )

print(
    f"Benchmark summary saved to '{SUMMARY_FILE}'."
)


# =============================================================================
# 10. Confusion Matrices
# =============================================================================

sns.set_theme(
    style="white",
    context="notebook",
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(18, 7),
)


# -----------------------------------------------------------------------------
# 10A. Intent Type Confusion Matrix
# -----------------------------------------------------------------------------

sns.heatmap(
    cm_type,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=TYPE_LABELS,
    yticklabels=TYPE_LABELS,
    ax=axes[0],
    annot_kws={
        "size": 12,
        "weight": "bold",
    },
)

axes[0].set_title(
    "Complexity Gate — Intent Type",
    fontsize=14,
    weight="bold",
    pad=14,
)

axes[0].set_xlabel(
    "Predicted Label",
    fontsize=11,
    weight="bold",
)

axes[0].set_ylabel(
    "True Label",
    fontsize=11,
    weight="bold",
)


# -----------------------------------------------------------------------------
# 10B. Primary Intent Confusion Matrix
# -----------------------------------------------------------------------------

sns.heatmap(
    cm_intent,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=SHORT_INTENT_LABELS,
    yticklabels=SHORT_INTENT_LABELS,
    ax=axes[1],
    annot_kws={
        "size": 10,
        "weight": "bold",
    },
)

axes[1].set_title(
    "Domain Routing — Primary Intent",
    fontsize=14,
    weight="bold",
    pad=14,
)

axes[1].set_xlabel(
    "Predicted Label",
    fontsize=11,
    weight="bold",
)

axes[1].set_ylabel(
    "True Label",
    fontsize=11,
    weight="bold",
)

axes[1].tick_params(
    axis="x",
    rotation=35,
)


# -----------------------------------------------------------------------------
# 10C. Figure Formatting
# -----------------------------------------------------------------------------

fig.suptitle(
    f"TypeSafe Jev System One — Benchmark Evaluation (N={N})",
    fontsize=16,
    weight="bold",
    y=1.02,
)

fig.text(
    0.5,
    0.01,
    (
        f"Intent Type Accuracy: {type_accuracy:.1%}  |  "
        f"Primary Intent Accuracy: {intent_accuracy:.1%}  |  "
        f"Strict Exact Accuracy: {strict_accuracy:.1%}"
    ),
    ha="center",
    fontsize=11,
)

plt.tight_layout()

plt.savefig(
    CM_PLOT_FILE,
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print(
    f"High-resolution confusion matrices saved to "
    f"'{CM_PLOT_FILE}'."
)


# =============================================================================
# 11. Final Summary
# =============================================================================

print("\n" + "=" * 70)
print("BENCHMARK COMPLETE")
print("=" * 70)

print(f"Queries Evaluated        : {N}")
print(f"Intent Type Accuracy     : {type_accuracy:.1%}")
print(f"Primary Intent Accuracy  : {intent_accuracy:.1%}")
print(f"Strict Exact Accuracy    : {strict_accuracy:.1%}")

if latency_stats:
    print(
        f"P50 Latency              : "
        f"{latency_stats['p50_ms']:.1f} ms"
    )
    print(
        f"P95 Latency              : "
        f"{latency_stats['p95_ms']:.1f} ms"
    )

print(f"Misclassified Queries    : {len(errors)}")
print(
    f"Error Rate               : "
    f"{len(errors) / N:.1%}"
)

print("\nGenerated Files:")
print(f"  {ERRORS_FILE}")
print(f"  {SUMMARY_FILE}")
print(f"  {CM_PLOT_FILE}")

Loaded 100/100 valid inference records.

1. INTENT TYPE — COMPLEXITY GATE
              precision    recall  f1-score   support

      single      1.000     0.982     0.991        55
    multiple      0.968     1.000     0.984        30
out_of_scope      1.000     1.000     1.000        15

    accuracy                          0.990       100
   macro avg      0.989     0.994     0.991       100
weighted avg      0.990     0.990     0.990       100


2. PRIMARY INTENT — DOMAIN ROUTING
                      precision    recall  f1-score   support

employee_information      1.000     1.000     1.000        11
              leaves      1.000     1.000     1.000        11
           timesheet      1.000     1.000     1.000        11
            benefits      1.000     0.909     0.952        11
             payroll      0.917     1.000     0.957        11
            multiple      0.967     0.967     0.967        30
        out_of_scope      1.000     1.000     1.000        15

           